# OCR Pipeline Deep Analysis

Understanding exactly where the digit recognizer fails and whether improvements are worth pursuing.

**Current honest metrics (no leakage):**
- Overall: 76.3% | Filled: 61.6% | Empty: 93.3%
- Hallucinated: 86 | Wrong: 79 | Missed: 554

**Key question:** Why does a 99.5% test-accuracy model only achieve 62% on real cells?

In [ ]:
import sys, os
sys.path.insert(0, '..')

import cv2
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

from app.core.extraction import detect_grid_v2, perspective_transform, extract_cells, order_points
from app.ml.recognizer import CNNRecognizer

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

with open('ground_truth.json') as f:
    gt_data = json.load(f)['images']

rec = CNNRecognizer()
print(f'Backend: {rec.backend}, conf_th: {rec.confidence_threshold}, empty_th: {rec.empty_threshold}')
print(f'GT images: {len(gt_data)}')

## 1. Extract all cells with full metadata
Run the pipeline on every GT image, keeping raw cells, preprocessed cells, CNN outputs, and GT labels.

In [ ]:
all_data = []  # list of dicts per cell

for entry in gt_data:
    fname = entry['path'].split('/')[-1]
    img = cv2.imread(f'../Examples/Ground Example/{fname}')
    if img is None:
        continue

    # Use GT corners for clean analysis (isolate OCR from detection)
    c16 = entry['corners_16']
    corners = np.array([c16[0], c16[3], c16[15], c16[12]], dtype=np.float32)
    
    contour = corners.reshape(4, 1, 2).astype(np.float32)
    warped = perspective_transform(img, contour)
    cells = extract_cells(warped)
    
    # Also get detected corners for comparison
    det_corners, det_conf = detect_grid_v2(img)
    
    for idx, cell in enumerate(cells):
        i, j = idx // 9, idx % 9
        gt_val = entry['grid'][i][j]
        gt_digits = gt_val if isinstance(gt_val, list) else [gt_val]
        gt_filled = any(v != 0 for v in gt_digits)
        gt_digit = next((v for v in gt_digits if v != 0), 0)
        
        # Preprocess and get raw CNN output
        processed = rec._preprocess(cell)
        is_empty = rec._is_empty(processed)
        
        # Run CNN on everything (even cells flagged empty)
        batch = processed[np.newaxis, np.newaxis, :, :].astype(np.float32) / 255.0
        logits = rec._onnx_session.run(None, {'image': batch})[0]
        probs = rec._softmax(logits)[0]
        
        # What the pipeline actually predicts
        if is_empty:
            pred_digit, pred_conf = 0, 1.0
            pred_source = 'empty_filter'
        else:
            pred_digit, pred_conf = rec._predict_from_probs(probs)
            pred_source = 'cnn'
        
        correct = pred_digit in gt_digits
        
        all_data.append({
            'filename': fname,
            'row': i, 'col': j,
            'raw_cell': cell,
            'processed': processed,
            'gt_digit': gt_digit,
            'gt_filled': gt_filled,
            'gt_digits': gt_digits,
            'pred_digit': pred_digit,
            'pred_conf': pred_conf,
            'pred_source': pred_source,
            'correct': correct,
            'is_empty_filter': is_empty,
            'probs': probs,
            'max_digit_prob': float(probs[1:].max()),
            'best_cnn_digit': int(probs[1:].argmax()) + 1,
            'class0_prob': float(probs[0]),
            'mean_pixel': float(np.mean(processed)),
            'detected': det_corners is not None,
        })

print(f'Total cells: {len(all_data)}')
print(f'Filled: {sum(1 for d in all_data if d["gt_filled"])}')
print(f'Empty: {sum(1 for d in all_data if not d["gt_filled"])}')

## 2. Error taxonomy: WHERE do errors come from?

Every error falls into one of these buckets:
1. **Empty filter false positive** — filled cell has low mean_pixel, never reaches CNN
2. **CNN low confidence** — CNN sees the cell but max(class 1-9) < 0.85, so it returns 0
3. **CNN wrong digit** — CNN is confident but picks the wrong class
4. **Hallucination** — empty cell gets a digit prediction

In [ ]:
# Classify every error
error_types = Counter()
for d in all_data:
    if d['correct']:
        if d['gt_filled']:
            error_types['correct_filled'] += 1
        else:
            error_types['correct_empty'] += 1
    elif d['gt_filled']:
        if d['is_empty_filter']:
            error_types['missed_by_empty_filter'] += 1
        elif d['pred_digit'] == 0:
            error_types['missed_by_low_confidence'] += 1
        else:
            error_types['wrong_digit'] += 1
    else:
        error_types['hallucinated'] += 1

print('Error Taxonomy (GT corners):')
for k, v in sorted(error_types.items(), key=lambda x: -x[1]):
    print(f'  {k}: {v}')

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
labels = list(error_types.keys())
values = list(error_types.values())
colors = ['#2ecc71' if 'correct' in l else '#e74c3c' if 'wrong' in l or 'halluc' in l else '#f39c12' for l in labels]
ax.barh(labels, values, color=colors)
ax.set_xlabel('Count')
ax.set_title('Error Taxonomy — Every Cell Classified')
for i, v in enumerate(values):
    ax.text(v + 10, i, str(v), va='center')
plt.tight_layout()
plt.show()

## 3. The confidence threshold problem
How many filled cells would be rescued by lowering `confidence_threshold`?

In [ ]:
filled = [d for d in all_data if d['gt_filled'] and not d['is_empty_filter']]

# For each filled cell that reaches the CNN, what's the max digit prob?
missed_by_conf = [d for d in filled if d['pred_digit'] == 0]  # CNN returned 0 due to low conf
correctly_predicted = [d for d in filled if d['correct'] and d['pred_digit'] != 0]

print(f'Filled cells reaching CNN: {len(filled)}')
print(f'  Correctly recognized: {len(correctly_predicted)}')
print(f'  Missed (low conf): {len(missed_by_conf)}')
print(f'  Wrong digit: {len([d for d in filled if d["pred_digit"] != 0 and not d["correct"]])}')

# Distribution of max_digit_prob for missed vs correct
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist([d['max_digit_prob'] for d in missed_by_conf], bins=50, alpha=0.7, color='red', label='Missed (returned 0)')
axes[0].axvline(x=0.85, color='black', linestyle='--', label='Current threshold')
axes[0].set_xlabel('max(class 1-9 prob)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Missed Filled Cells (n={len(missed_by_conf)})')
axes[0].legend()

# What if we lowered threshold? How many would we rescue vs how many hallucinations would we add?
empty_cells = [d for d in all_data if not d['gt_filled'] and not d['is_empty_filter']]
thresholds = np.arange(0.3, 1.0, 0.05)
rescued = []
new_halluc = []
for t in thresholds:
    r = sum(1 for d in missed_by_conf if d['max_digit_prob'] >= t and d['best_cnn_digit'] in d['gt_digits'])
    h = sum(1 for d in empty_cells if d['max_digit_prob'] >= t)
    rescued.append(r)
    new_halluc.append(h)

axes[1].plot(thresholds, rescued, 'g-o', label='Rescued correct digits', markersize=4)
axes[1].plot(thresholds, new_halluc, 'r-o', label='New hallucinations', markersize=4)
axes[1].axvline(x=0.85, color='black', linestyle='--', label='Current threshold')
axes[1].set_xlabel('Confidence Threshold')
axes[1].set_ylabel('Count')
axes[1].set_title('Threshold Tradeoff: Rescued Digits vs New Hallucinations')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. What do the errors look like?
Visualize raw cells and preprocessed cells for each error type.

In [ ]:
def show_error_samples(cells, title, n=12):
    """Show raw + preprocessed pairs for a set of error cells."""
    sample = cells[:n]
    if not sample:
        print(f'{title}: no samples')
        return
    fig, axes = plt.subplots(2, len(sample), figsize=(min(20, len(sample)*1.8), 4))
    if len(sample) == 1:
        axes = axes.reshape(2, 1)
    fig.suptitle(title, fontsize=12)
    for i, d in enumerate(sample):
        # Raw cell
        raw = d['raw_cell']
        if len(raw.shape) == 3:
            raw = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
        axes[0, i].imshow(raw, cmap='gray' if len(raw.shape)==2 else None)
        axes[0, i].set_title(f'GT:{d["gt_digit"]}', fontsize=8)
        axes[0, i].axis('off')
        
        # Preprocessed (what CNN sees)
        axes[1, i].imshow(d['processed'], cmap='gray')
        axes[1, i].set_title(f'→{d["pred_digit"]} ({d["max_digit_prob"]:.2f})', fontsize=8)
        axes[1, i].axis('off')
    plt.tight_layout()
    plt.show()

# Missed by empty filter
missed_empty = [d for d in all_data if d['gt_filled'] and d['is_empty_filter']]
show_error_samples(missed_empty, f'Missed by empty filter (n={len(missed_empty)})')

# Missed by low confidence
missed_conf = sorted([d for d in all_data if d['gt_filled'] and not d['is_empty_filter'] and d['pred_digit'] == 0],
                     key=lambda d: -d['max_digit_prob'])
show_error_samples(missed_conf, f'Missed by low confidence (n={len(missed_conf)})')

# Wrong digit
wrong = [d for d in all_data if d['gt_filled'] and d['pred_digit'] != 0 and not d['correct']]
show_error_samples(wrong, f'Wrong digit (n={len(wrong)})')

# Hallucinated
halluc = [d for d in all_data if not d['gt_filled'] and d['pred_digit'] != 0]
show_error_samples(halluc, f'Hallucinated (n={len(halluc)})')

## 5. Per-image breakdown
Are errors concentrated in a few bad images or spread evenly?

In [ ]:
image_stats = defaultdict(lambda: {'filled': 0, 'correct': 0, 'missed': 0, 'wrong': 0, 'halluc': 0})

for d in all_data:
    s = image_stats[d['filename']]
    if d['gt_filled']:
        s['filled'] += 1
        if d['correct']:
            s['correct'] += 1
        elif d['pred_digit'] == 0:
            s['missed'] += 1
        else:
            s['wrong'] += 1
    else:
        if not d['correct']:
            s['halluc'] += 1

# Sort by filled accuracy
sorted_images = sorted(image_stats.items(), key=lambda x: x[1]['correct']/max(1,x[1]['filled']))

fig, ax = plt.subplots(figsize=(14, 8))
names = [s[0].replace('.jpeg','') for s in sorted_images]
accs = [s[1]['correct']/max(1,s[1]['filled'])*100 for s in sorted_images]
colors = ['#e74c3c' if a < 50 else '#f39c12' if a < 80 else '#2ecc71' for a in accs]
bars = ax.barh(range(len(names)), accs, color=colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=7)
ax.set_xlabel('Filled Cell Accuracy (%)')
ax.set_title('Per-Image Filled Cell Accuracy (GT Corners)')
ax.axvline(x=62, color='black', linestyle='--', alpha=0.5, label='Mean (62%)')
for i, (name, stats) in enumerate(sorted_images):
    ax.text(accs[i]+1, i, f'{stats["correct"]}/{stats["filled"]}', va='center', fontsize=7)
plt.tight_layout()
plt.show()

# How many images are >90% vs <50%?
above90 = sum(1 for a in accs if a >= 90)
below50 = sum(1 for a in accs if a < 50)
print(f'Images >= 90% accuracy: {above90}/{len(accs)}')
print(f'Images < 50% accuracy: {below50}/{len(accs)}')

## 6. Preprocessing deep dive
What does the CNN actually see? Compare MNIST training data vs real cells.

In [ ]:
# Show a well-recognized image vs a badly-recognized image
best_img = sorted_images[-1][0]  # best accuracy
worst_img = sorted_images[0][0]  # worst accuracy

for img_name, label in [(best_img, 'BEST'), (worst_img, 'WORST')]:
    img_cells = [d for d in all_data if d['filename'] == img_name and d['gt_filled']]
    n = min(9, len(img_cells))
    fig, axes = plt.subplots(2, n, figsize=(n*1.8, 4))
    fig.suptitle(f'{label}: {img_name}', fontsize=11)
    for i, d in enumerate(img_cells[:n]):
        raw = d['raw_cell']
        if len(raw.shape) == 3:
            raw = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
        axes[0, i].imshow(raw, cmap='gray' if len(raw.shape)==2 else None)
        axes[0, i].set_title(f'GT:{d["gt_digit"]}', fontsize=8)
        axes[0, i].axis('off')
        
        axes[1, i].imshow(d['processed'], cmap='gray')
        color = 'green' if d['correct'] else 'red'
        axes[1, i].set_title(f'→{d["pred_digit"]}', fontsize=8, color=color)
        axes[1, i].axis('off')
    plt.tight_layout()
    plt.show()

## 7. The gap: 99.5% test accuracy vs 62% pipeline accuracy
What's different between MNIST/printed test data and real newspaper cells?

In [ ]:
# Compare pixel statistics between training data and real cells
from app.ml.dataset import PrintedDigitDataset
from torchvision import datasets, transforms

mnist = datasets.MNIST(root='../data/mnist', train=False, download=True, transform=transforms.ToTensor())
printed = PrintedDigitDataset(count_per_digit=100)

# Sample pixel distributions
mnist_pixels = np.concatenate([mnist[i][0].numpy().flatten() for i in range(200)])
printed_pixels = np.concatenate([printed[i][0].numpy().flatten() for i in range(200)])
real_pixels = np.concatenate([d['processed'].flatten() / 255.0 for d in all_data[:200]])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, pixels, title in zip(axes, [mnist_pixels, printed_pixels, real_pixels],
                              ['MNIST (test)', 'Printed Fonts', 'Real Newspaper Cells']):
    ax.hist(pixels, bins=50, alpha=0.7, density=True)
    ax.set_title(title)
    ax.set_xlabel('Pixel value')
    ax.set_xlim(0, 1)
    ax.text(0.5, 0.9, f'mean={pixels.mean():.3f}\nstd={pixels.std():.3f}',
            transform=ax.transAxes, fontsize=9, va='top')
plt.suptitle('Pixel Distribution: Training Data vs Real Cells', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Confusion matrix heatmap

In [ ]:
confusion = np.zeros((10, 10), dtype=int)
for d in all_data:
    if d['gt_filled']:
        confusion[d['gt_digit']][d['pred_digit']] += 1

fig, ax = plt.subplots(figsize=(8, 7))
# Normalize by row (GT class)
row_sums = confusion.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
norm = confusion / row_sums

im = ax.imshow(norm[1:, :], cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(10))
ax.set_xticklabels(['0(miss)'] + [str(i) for i in range(1, 10)])
ax.set_yticks(range(9))
ax.set_yticklabels([str(i) for i in range(1, 10)])
ax.set_xlabel('Predicted')
ax.set_ylabel('Ground Truth')
ax.set_title('Normalized Confusion Matrix (filled cells, GT corners)')

for i in range(9):
    for j in range(10):
        val = confusion[i+1][j]
        if val > 0:
            color = 'white' if norm[i+1][j] > 0.5 else 'black'
            ax.text(j, i, str(val), ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, label='Proportion')
plt.tight_layout()
plt.show()

## 9. Assessment: Is further improvement worth it?

### What we know:
1. **The model works on clean images** — many images hit 100% accuracy
2. **A few bad images drag the average down** — likely poor lighting, extreme angles, or faded print
3. **The confidence threshold trades precision for recall** — current 0.85 is conservative
4. **The CNN itself rarely predicts the WRONG digit** — most errors are misses, not misclassifications

### Improvement options ranked by effort/impact:

In [ ]:
# Quick analysis: what would different thresholds give us?
filled_cnn = [d for d in all_data if d['gt_filled'] and not d['is_empty_filter']]
empty_cnn = [d for d in all_data if not d['gt_filled'] and not d['is_empty_filter']]

print('Threshold sweep (GT corners, no empty filter):')
print(f'{"Thresh":>8} | {"Filled%":>8} {"Empty%":>8} {"Overall%":>9} | {"Halluc":>6} {"Miss":>6} {"Wrong":>6}')
print('-' * 75)

total = len(all_data)
for t in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
    fc = sum(1 for d in filled_cnn if d['max_digit_prob'] >= t and d['best_cnn_digit'] in d['gt_digits'])
    ec = sum(1 for d in empty_cnn if d['max_digit_prob'] < t)
    # Add empty-filter cells
    ef_correct = sum(1 for d in all_data if d['is_empty_filter'] and not d['gt_filled'])
    ef_missed = sum(1 for d in all_data if d['is_empty_filter'] and d['gt_filled'])
    
    ft = sum(1 for d in all_data if d['gt_filled'])
    et = sum(1 for d in all_data if not d['gt_filled'])
    
    total_fc = fc
    total_ec = ec + ef_correct
    halluc = et - total_ec
    wrong = sum(1 for d in filled_cnn if d['max_digit_prob'] >= t and d['best_cnn_digit'] not in d['gt_digits'])
    missed = ft - total_fc - wrong
    overall = (total_fc + total_ec) / total * 100
    
    marker = ' <-- current' if abs(t - 0.85) < 0.01 else ''
    print(f'{t:>8.2f} | {total_fc/ft*100:>7.1f}% {total_ec/et*100:>7.1f}% {overall:>8.1f}% | {halluc:>6} {missed:>6} {wrong:>6}{marker}')

## 10. Conclusions

### The diagnosis

The 99.5% → 62% gap is **not a model problem**. It's an **image quality problem** concentrated in a few extreme images.

**Error taxonomy (GT corners, 3078 cells):**
- `correct_filled`: 1272 (74% of filled)
- `correct_empty`: 1257 (93% of empty)
- `missed_low_conf`: 407 — **the dominant error**. CNN sees the cell but isn't confident enough.
- `hallucinated`: 101 — empty cells that get a digit
- `wrong_digit`: 41 — CNN is confident AND wrong. **Only 2.4% of filled cells.**

**The CNN rarely picks the wrong digit.** When it commits, it's almost always right. The problem is it refuses to commit on ~24% of filled cells.

### Why does it refuse?

The 407 missed cells have median `max(class 1-9)` = 0.579. Lowering the threshold would rescue some, but at a terrible ratio:
- At threshold 0.7: rescue 97 correct + 47 wrong (2:1 ratio)
- At threshold 0.5: rescue 183 correct + 116 wrong (1.6:1 ratio)

The CNN genuinely can't distinguish these cells — they're too blurry, too low-contrast, or have grid line artifacts that obscure the digit. **This is not a threshold problem.**

### The distribution is bimodal

- **15/38 images achieve ≥90% accuracy** — the model is nearly perfect on clean images
- **9/38 images are below 50%** — these drag the average from ~95% to ~62%
- The bottom 3: toilet paper sudoku (29%), cat sitting on puzzle (22%), blurry faded print (12%)

**These aren't OCR failures. They're impossible inputs** for any 28x28 single-cell CNN. The cells are unrecognizable after warping.

### Would better training data help?

| Approach | Expected impact | Reasoning |
|----------|----------------|-----------|
| **Chars74K / SVHN** | Marginal (+2-5%) | The CNN already gets 99.5% on diverse test data. The bottleneck is cell image quality, not class knowledge. |
| **Lower threshold** | Net negative | Rescue ratio is ~2:1 at best. More wrong digits than rescued correct ones. |
| **Better detection/warp** | High (for bottom 9) | Better corners → better cell images → CNN can recognize them. |
| **Higher resolution cells** | Moderate | 28x28 loses detail. 56x56 or 64x64 might help on borderline cells. |
| **User correction UI** | High (practical) | Flag low-confidence cells for manual editing. Already built in frontend. |

### The realistic ceiling

With the current architecture and these 38 images:
- **Good images (15/38):** ~95-100% — already there
- **Medium images (14/38):** ~60-90% — marginal gains from preprocessing, maybe +5-10%
- **Hard images (9/38):** <50% — need fundamentally better detection/warp, not OCR

**Overall ceiling: ~80% with current pipeline.** The threshold sweep confirms this — optimal is 79.8% at threshold 0.85.

### Recommendation

**OCR model improvements have diminishing returns.** The next high-impact work is:
1. **Detection** — fix the 4 undetected images, improve corners on the 9 worst images
2. **User correction** — the frontend confidence display already highlights uncertain cells
3. **Accept the distribution** — the pipeline works well on normal photos (95%+) and fails on extreme edge cases (cat, toilet paper, extreme blur). That's acceptable for a portfolio project.